In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

token = os.environ.get('HUGGINGFACE_ACCESS_TOKEN')

In [5]:
import whisperx

device = "cuda" 
audio_file = "sample.wav"
batch_size = 16 # reduce if low on GPU mem
compute_type = "float16" # change to "int8" if low on GPU mem (may reduce accuracy)

# save model to local path (optional)
model_dir = "./data"
model = whisperx.load_model("large-v2", device, compute_type=compute_type, download_root=model_dir)

tokenizer.json:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.80k [00:00<?, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

vocabulary.txt:   0%|          | 0.00/460k [00:00<?, ?B/s]

Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.2.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/torch/whisperx-vad-segmentation.bin`


No language specified, language will be first be detected for each audio file (increases inference time).
Model was trained with pyannote.audio 0.0.1, yours is 3.1.1. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.2.1+cu121. Bad things might happen unless you revert torch to 1.x.


In [6]:

audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)
print(result["segments"]) # before alignment

Detected language: en (1.00) in first 30s of audio...
[{'text': " 20 years ago, I didn't have a phone. Or, you know, 20, well, 24 years ago I didn't have a phone. And I was fine. Everybody was fine. Look how happy you were. You were so successful. I was happy. And then it was only 15 years ago that I didn't have a smartphone. I had a flip phone. That was fine. I was fine with that.", 'start': 0.009, 'end': 19.292}, {'text': " But ever since I had a ever since I had a smartphone, you know, but nowadays my god you go You know three hours without a smartphone and the whole thing goes to shit Jason The world changed a little too John I mean it did but you know I can understand like like last year Jason Finn asked me to go to a baseball game and On my way there. I was on the train on the way to the baseball stadium. I realized I didn't have my phone and", 'start': 19.292, 'end': 44.855}, {'text': ' Did I tell you this story? I end up at the baseball stadium, and on my phone should be my tic

In [7]:

# 2. Align whisper output
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

print(result["segments"]) # after alignment

[{'start': 0.349, 'end': 2.332, 'text': " 20 years ago, I didn't have a phone.", 'words': [{'word': '20'}, {'word': 'years', 'start': 0.389, 'end': 0.67, 'score': 0.612}, {'word': 'ago,', 'start': 0.73, 'end': 0.99, 'score': 0.916}, {'word': 'I', 'start': 1.01, 'end': 1.03, 'score': 0.0}, {'word': "didn't", 'start': 1.571, 'end': 1.751, 'score': 0.835}, {'word': 'have', 'start': 1.791, 'end': 1.911, 'score': 0.743}, {'word': 'a', 'start': 1.951, 'end': 1.971, 'score': 0.978}, {'word': 'phone.', 'start': 2.031, 'end': 2.332, 'score': 0.916}]}, {'start': 3.013, 'end': 6.997, 'text': "Or, you know, 20, well, 24 years ago I didn't have a phone.", 'words': [{'word': 'Or,', 'start': 3.013, 'end': 3.173, 'score': 0.752}, {'word': 'you', 'start': 3.253, 'end': 3.333, 'score': 0.862}, {'word': 'know,', 'start': 3.373, 'end': 3.493, 'score': 0.916}, {'word': '20,'}, {'word': 'well,', 'start': 4.614, 'end': 4.835, 'score': 0.914}, {'word': '24'}, {'word': 'years', 'start': 5.616, 'end': 5.796, 's

In [9]:
# 3. Assign speaker labels
diarize_model = whisperx.DiarizationPipeline(use_auth_token=token, device=device)

# add min/max number of speakers if known
diarize_segments = diarize_model(audio, min_speakers=2, max_speakers=2)

result = whisperx.assign_word_speakers(diarize_segments, result)
print(diarize_segments)
print(result["segments"]) # segments are now assigned speaker IDs

                              segment label     speaker       start  \
0   [ 00:00:00.008 -->  00:00:09.957]     A  SPEAKER_00    0.008489   
1   [ 00:00:01.349 -->  00:00:01.587]     B  SPEAKER_01    1.349745   
2   [ 00:00:01.604 -->  00:00:01.621]     C  SPEAKER_01    1.604414   
3   [ 00:00:07.410 -->  00:00:07.597]     D  SPEAKER_01    7.410866   
4   [ 00:00:09.142 -->  00:00:09.940]     E  SPEAKER_01    9.142615   
..                                ...   ...         ...         ...   
86  [ 00:05:08.378 -->  00:05:09.006]    CI  SPEAKER_01  308.378608   
87  [ 00:05:09.006 -->  00:05:09.804]    CJ  SPEAKER_00  309.006791   
88  [ 00:05:09.804 -->  00:05:09.838]    CK  SPEAKER_01  309.804754   
89  [ 00:05:10.806 -->  00:05:21.485]    CL  SPEAKER_00  310.806452   
90  [ 00:05:15.628 -->  00:05:16.052]    CM  SPEAKER_01  315.628183   

           end  intersection       union  
0     9.957555   -311.243445  321.452511  
1     1.587436   -319.613564  320.111255  
2     1.621392   -